<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/credit_risk_ccar_vecm_lgbm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Systemic Credit Portfolio Macro-Transmission Engine (VECM + Monotonic Machine Learning**

### **Regulatory Context**

Under the Federal Reserve's Comprehensive Capital Analysis and Review (CCAR) and DFAST regulatory stress testing regimes, tier-1 financial institutions must model credit portfolio losses across a 9-quarter forecast horizon under severe macroeconomic stress scenarios (e.g., Severly Adverse).

A critical vulnerability in traditional credit risk modeling is the failure to enforece **economic monotonicity** and structural macroeconomic cointegration:

1. **Cointegrated Macro Dynamics**: Key macro variables - such as Gross Domestic Product (GDP), Unemployment Rate (UR), Commerial Real Estate Price Index (CRE), and short-term interest rates (SOFR) - share long-run equilibrium relationships. Unrestricted Vector Autoregressions (VAR) drift into economically impossible regimes. A Vector Error Correction Model (VECM) captures short-run dynamics while anchoring long-run cointegrating vectors.

2. **Economic Monotonicity Constraints**: Standard unconstrained Machine Learning models (e.g., Random Forests, standard GBDTs, Deep Neural Networks) often exhibit non-monotonic artifacts due to overfitting or sparse training regions (e.g., predicting lower default rates when unemployment surges from 8% to 11%). Under Fed SR 11-7 / OCC 2011-12 model validation standards, such behavior triggers severe audit findings. Monotonic Gradient Boosted Decision Trees (LightGBM) strictly enforce partial derivative contraints:

$$\frac{\partial P(\text{Default})}{\partial \text{Unemployment}} \ge 0, \quad \frac{\partial P(\text{Default})}{\partial \text{GDP Growth}} \le 0, \quad \frac{\partial P(\text{Default})}{\partial \text{CRE Index}} \le 0$$

### **Mathematical Methodology**

1. Cointegrated Macro System (VECM)

Let $Y_{t} = [UR_{t}, \Delta GDP_{t}, \Delta CRE_{t}, SOFR_{t}]^{T}$ be an $n \times 1$ vector of non-stationary $I(1)$ macro variables. The VECM specification is given by:

$$\Delta Y_{t} = \Pi Y_{t-1} + \sum^{p-1}_{i=1} \Gamma_{i}\Delta Y_{t-i} + \mu + \epsilon_{t}, \quad \epsilon_{t} \sim N(0, \Sigma)$$

where $\Pi = \alpha \beta^{T}$ is reduced to rank $r < n$, with $\beta$ containing the $r$ cointegrating vectors (long-run equilibrium) and $\alpha$ representing the speed-of-adjustment matrix.

2. Monotonically Constrained Credit Default Model
To project quarterly Default Probabilities ($PD_{i,t}$), we fit a LightGBM regressor over macroeconomic factors $X_{t}$ with explicit constraint vector $M = [m_{1}, m_{2}, ... , m_{k}] \in \{-1, 0, 1\}^{k}$

$$LGD_{i,t} = \text{min}\left(\text{max}\left( LGD_{i,0}\cdot\left(\frac{CRE_{0}}{CRE_{t}}\right)^{\eta}, LGD_{i,0}\right), 0.90\right)$$

where $\eta > 0$ is the collateral elasticity parameter. The total portfolio Expected Loss ($EL_{t}$) at quarter $t$ across $N$ facility exposures ($EAD_{i}$) is:
$$EL_{t}=\sum^{N}_{i=1}EAD_{i,t}\cdot PD_{i,t}(Y_{t})\cdot LGD_{i,t}(Y_{t})$$

In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List

# Set seed for reproducible institutional benchmark
np.random.seed(42)

# ==========================================================================
# 1. SYNTHETIC DATA GENERATION ENGINE
# ==========================================================================
def generate_macro_historical_data(n_quarters: int = 80) -> pd.DataFrame:
  """Generates 20 years of quarterly macro data with cointegration relationships."""
  dates = pd.date_range(start="2006-03-31", periods=n_quarters, freq="QE")

  # Random walk drivers
  e1 = np.random.normal(0, 0.4, n_quarters)
  e2 = np.random.normal(0, 0.5, n_quarters)

  unemp = np.zeros(n_quarters)
  gdp = np.zeros(n_quarters)
  cre_index = np.zeros(n_quarters)
  sofr = np.zeros(n_quarters)

  unemp[0], gdp[0], cre_index[0], sofr[0] = 5.0, 2.5, 100.0, 4.0

  for t in range(1, n_quarters):
    # Long-run cointegrating relationships: CRE values drop when unemployment surges
    unemp[t] = np.clip(unemp[t-1] + 0.3 * (unemp[t-1] - 5.0) * 0.1 + e1[t], 3.0, 12.0)
    gdp[t] = 3.0 - 0.6 * (unemp[t] - 5.0) + np.random.normal(0, 0.8)
    cre_index[t] = np.maximum(cre_index[t-1] * (1.0 + 0.01 * (gdp[t] - 1.5) - 0.02 * (unemp[t] - 5.0) + e2[t]*0.02), 50.0)
    sofr[t] = np.clip(0.8 * sofr[t-1] + 0.2 * (gdp[t] + 2.0) + np.random.normal(0, 0.3), 0.25, 8.0)

  df = pd.DataFrame({
      'unemployment_rate': unemp,
      'gdp_growth': gdp,
      'cre_index': cre_index,
      'sofr_rate': sofr
  }, index=dates)

  return df

def generate_loan_portfolio(n_loans: int = 5000) -> pd.DataFrame:
  """Generates synthetic corporate CRE/Wholesale loan book."""
  loans = pd.DataFrame({
      'loan_id': [f"LN_{i:06d}" for i in range(1, n_loans + 1)],
      'ead': np.random.lognormal(mean=14.5, sigma=1.0, size=n_loans), # ~$2M avg exposure
      'base_lgd': np.random.uniform(0.35, 0.45, size=n_loans),
      'base_pd': np.random.beta(a=2, b=50, size=n_loans), # Avg ~3.8% baseline PD
      'sector': np.random.choice(['CRE_Office', 'CRE_Retail', 'Corporate_Industrial'], size=n_loans)
  })
  return loans

# ==========================================================================
# 2. VECM MACROECONOMIC ENGINE
# ==========================================================================
class StructuralVECMEngine:
  """Vector Error Correction Model for CCAR Macro Projection."""
  def __init__(self, data: pd.DataFrame):
    self.data = data
    self.model = None
    self.results = None

  def fit(self, k_ar_diff: int = 1):
    """Fits VECM after Johansen Cointegration test rank determination."""
    # Check Johansen rank
    rank_test = select_coint_rank(self.data, det_order=0, k_ar_diff=k_ar_diff, method="trace")
    c_rank = max(rank_test.rank, 1)

    self.model = VECM(self.data, k_ar_diff=k_ar_diff, coint_rank=c_rank, deterministic="co")
    self.results = self.model.fit()

  def forecast(self, steps: int = 9) -> pd.DataFrame:
    """Projects macro paths over 9 quarters (CCAR horizon)."""
    forecast_array = self.results.predict(steps=steps)
    dates = pd.date_range(start=self.data.index[-1] + pd.DateOffset(months=3), periods=steps, freq="QE")
    return pd.DataFrame(forecast_array, index=dates, columns=self.data.columns)

# ==========================================================================
# 3. MONOTONIC LIGHTGBM DEFAULT MODEL
# ==========================================================================
class MonotonicCreditLossEngine:
  """
  LightGBM Default Predictor with Strict Regulatory Monotonic Constraints.
  """
  def __init__(self):
    self.model = None
    self.feature_names = ['unemployment_rate', 'gdp_growth', 'cre_index', 'sofr_rate']

  def train_constrained_model(self, macro_df: pd.DataFrame):
    """
    Trains LightGBM model enforcing monotonicity:
    - Unemployment Rate: Positive (+1)
    - GDP Growth: Negative (-1)
    - CRE Index: Negative (-1)
    - SOFR Rate: Positive (+1)
    """
    n = len(macro_df)
    # Synthetic historical default rates tied to macro
    y_default_rate = (
        0.005
        + 0.006 * macro_df['unemployment_rate']
        - 0.003 * macro_df['gdp_growth']
        - 0.0002 * macro_df['cre_index']
        + 0.002 * macro_df['sofr_rate']
        + np.random.normal(0, 0.002, n)
    )
    y_default_rate = np.clip(y_default_rate, 0.001, 0.25)

    # Monotonicity specs: 1 = Increasing, -1 = Decreasing, 0 = Unconstrained
    monotone_constraints = [1, -1, -1, 1]

    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'learning_rate': 0.03,
        'num_leaves': 12,
        'monotone_constraints': monotone_constraints,
        'verbose': -1,
        'min_child_samples': 5
    }

    dtrain = lgb.Dataset(macro_df[self.feature_names], label=y_default_rate)
    self.model = lgb.train(params, dtrain, num_boost_round=150)

  def predict_portfolio_loss(self, scenario_macro: pd.DataFrame, loans: pd.DataFrame) -> pd.DataFrame:
    """Projects portfolio Expected Loss (EL) and dynamic LGD per scenario quarter."""
    macro_pds = self.model.predict(scenario_macro[self.feature_names])
    base_cre = scenario_macro['cre_index'].iloc[0]

    results = []
    for t, (idx, macro_row) in enumerate(scenario_macro.iterrows()):
      q_pd_multiplier = macro_pds[t] / macro_pds[0] # Scale relative to baseline

      # Collateral Haircut Impact on LGD
      cre_shock_ratio = base_cre / macro_row['cre_index']

      # Dynamic LGD adjustment based on collateral impairment
      dynamic_lgd = np.clip(loans['base_lgd'] * (cre_shock_ratio ** 0.5), loans['base_lgd'], 0.90)
      dynamic_pd = np.clip(loans['base_pd'] * q_pd_multiplier, 0.0001, 0.95)

      quarter_el = np.sum(loans['ead'] * dynamic_pd * dynamic_lgd)
      total_ead = np.sum(loans['ead'])

      results.append({
          'Quarter': f"{idx.year}-Q{idx.quarter}",
          'Unemployment': macro_row['unemployment_rate'],
          'CRE_Index': macro_row['cre_index'],
          'Avg_Portfolio_PD': np.mean(dynamic_pd),
          'Avg_Portfolio_LGD': np.mean(dynamic_lgd),
          'Total_Expected_Loss_$': quarter_el,
          'Loss_Rate_Bps': (quarter_el / total_ead) * 10000
      })

    return pd.DataFrame(results)


# ==========================================================================
# 4. EXECUTION & CCAR STRESS PIPELINE
# ==========================================================================
if __name__ == "__main__":
  # 1. Load Data
  macro_hist = generate_macro_historical_data(n_quarters=80)
  loan_portfolio = generate_loan_portfolio(n_loans=5000)

  # 2. Fit Macro VECM Engine
  vecm_engine = StructuralVECMEngine(macro_hist)
  vecm_engine.fit(k_ar_diff=1)
  baseline_projection = vecm_engine.forecast(steps=9)

  # 3. Create CCAR Severely Adverse Scenario Override
  severely_adverse_macro = baseline_projection.copy()
  # Shock: Unemployment spikes +4.5%, CRE drops 25%, GDP contracts
  severely_adverse_macro['unemployment_rate'] += np.linspace(1.0, 5.0, 9)
  severely_adverse_macro['gdp_growth'] -= np.linspace(2.0, 5.0, 9)
  severely_adverse_macro['cre_index'] *= np.linspace(0.95, 0.70, 9)
  severely_adverse_macro['sofr_rate'] = np.maximum(severely_adverse_macro['sofr_rate'] - np.linspace(0.5, 2.5, 9), 0.25)

  # 4. Fit Monotonic Credit Loss Engine & Run Projections
  loss_engine = MonotonicCreditLossEngine()
  loss_engine.train_constrained_model(macro_hist)

  baseline_loss_df = loss_engine.predict_portfolio_loss(baseline_projection, loan_portfolio)
  adverse_loss_df = loss_engine.predict_portfolio_loss(severely_adverse_macro, loan_portfolio)

  print(" === CCAR 9-QUARTER SEVERLELY ADVERSE STRESS RESULTS ===")
  print(adverse_loss_df[['Quarter', 'Unemployment', 'CRE_Index', 'Avg_Portfolio_PD', 'Loss_Rate_Bps']].to_string(index=False))


 === CCAR 9-QUARTER SEVERLELY ADVERSE STRESS RESULTS ===
Quarter  Unemployment   CRE_Index  Avg_Portfolio_PD  Loss_Rate_Bps
2026-Q1      4.876858 2227.248376          0.038374     156.411783
2026-Q2      5.491090 2246.908839          0.080880     329.664557
2026-Q3      6.096216 2261.643935          0.108814     443.524117
2026-Q4      6.682647 2273.083497          0.154067     627.973837
2027-Q1      7.274408 2280.846430          0.154067     627.973837
2027-Q2      7.871883 2284.694731          0.152919     623.292579
2027-Q3      8.473831 2284.410486          0.152919     623.292579
2027-Q4      9.080787 2279.722821          0.152919     623.292579
2028-Q1      9.693170 2270.340136          0.152919     623.292579


In [11]:
# ==========================================================================
# 5. Visualization Engine Implementation
# ==========================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# 1. Load exact run results
data = {
    'Quarter': ['2026-Q1', '2026-Q2', '2026-Q3', '2026-Q4', '2027-Q1', '2027-Q2', '2027-Q3', '2027-Q4', '2028-Q1'],
    'Unemployment': [4.876858, 5.491090, 6.096216, 6.682647, 7.274408, 7.871883, 8.473831, 9.080787, 9.693170],
    'CRE_Index': [2227.248376, 2246.908839, 2261.643935, 2273.083497, 2280.846430, 2284.694731, 2284.410486, 2279.722821, 2270.340136],
    'Avg_Portfolio_PD': [0.038374, 0.080880, 0.108814, 0.154067, 0.154067, 0.152919, 0.152919, 0.152919, 0.152919],
    'Loss_Rate_Bps': [156.411783, 329.664557, 443.524117, 627.973837, 627.973837, 623.292579, 623.292579, 623.292579, 623.292579]
}

df = pd.DataFrame(data)
df['Avg_Portfolio_PD_Pct'] = df['Avg_Portfolio_PD'] * 100
df['Cumulative_Loss_Bps'] = df['Loss_Rate_Bps'].cumsum()

# 2. Create Subplot Dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "<b>Macro Scenario: Unemployment vs CRE Index</b>",
        "<b>Credit Risk Escalation: Portfolio PD (%)</b>",
        "<b>Quarterly Loss Rate (Bps)</b>",
        "<b>Cumulative Loss Accumulation (Bps)</b>"
    ),
    specs=[[{"secondary_y": True}, {}], [{}, {}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.10
)

# Color Palette
c_navy = "#1f77b4"
c_red = "#d62728"
c_amber = "#ff7f0e"
c_purple = "#9467bd"

# Subplot 1: Macro Factors (Dual Axis)
fig.add_trace(
    go.Scatter(x=df['Quarter'], y=df['Unemployment'], name="Unemployment Rate (%)", line=dict(color=c_red, width=3)),
    row=1, col=1, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=df['Quarter'], y=df['CRE_Index'], name="CRE Index", line=dict(color=c_navy, width=2, dash='dot')),
    row=1, col=1, secondary_y=True
)

# Subplot 2: Portfolio Average PD
fig.add_trace(
    go.Scatter(x=df['Quarter'], y=df['Avg_Portfolio_PD_Pct'], name="Avg PD (%)", line=dict(color=c_amber, width=3), mode='lines+markers'),
    row=1, col=2
)

# Subplot 3: Quarterly Loss Rate (Bar Chart)
fig.add_trace(
    go.Bar(x=df['Quarter'], y=df['Loss_Rate_Bps'], name="Loss Rate (Bps)", marker_color=c_red, opacity=0.8),
    row=2, col=1
)

# Subplot 4: Cumulative Loss Accumulation (Area Chart)
fig.add_trace(
    go.Scatter(x=df['Quarter'], y=df['Cumulative_Loss_Bps'], name="Cumulative Loss (Bps)", fill='tozeroy', line=dict(color=c_purple, width=2)),
    row=2, col=2
)

# Update Axes & Layout
fig.update_xaxes(title_text="CCAR Forecast Quarter", gridcolor="gainsboro")
fig.update_yaxes(title_text="Unemployment (%)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="CRE Index", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="Default Prob. (%)", row=1, col=2)
fig.update_yaxes(title_text="Loss Rate (Bps)", row=2, col=1)
fig.update_yaxes(title_text="Cumulative Loss (Bps)", row=2, col=2)

fig.update_layout(
    title_text="<b>CCAR 9-Quarter Severely Adverse Stress Testing Dashboard</b><br><sup>Institutional Risk Analytics | Monotonic LightGBM + VECM Engine</sup>",
    title_font_size=20,
    height=800,
    width=1200,
    template="plotly_white",
    showlegend=False
)

# Render Dashboard
fig.show()

# **Summary**

### **1. Result Summary**
This report evaluates the credit risk and capital depletion dynamics for the wholesale loan portfolio under the simulated 9-quarter CCAR Severely Adverse Scenario (2026-Q1 to 2028-Q2).

Across the 9-quarter forecast horizon, the severe macroeconomic shock drives portfolio Average Probability of Default ($PD$) from 3.84% in 2026-Q1 to a peak of 15.41% in 2026-Q4. Quarterly loss rates surge from 156.41 bps to a peak of 627.97 bps, generating a cumulative 9-quarter portfolio loss of 46.79% total cumulative credit loss.

### **2. Model Validation & Econometric Deep-Dive**
#### **A. Non-Linear Loss Saturation & Leaf Boundary Plateau**
The numerical output reveals a key structural pattern: between 2026-Q4 (Unemployment = 6.68%) and 2028-Q1 (Unemployment = 9.69%), $PD$ plateaus precisely at ~15.29% - 15.51%, and quarterly loss rates flatten out at ~ 623 - 628 bps.
* **Root Cause Analysis**: LightGBM decision tree partition feature space based on historical training bounds. When the stress scenario pushes unemployment beyond ~ 6.7% into unobserved tail territory, the decision tree encounters its terminal leaf boundary.
* **Model Risk Finding**: Under Fed SR 11-7 / OCC 2011-12 guidelines, model validators will note that non-parametric tree ensembles cap predictions at extreme out-of-sample regions rather than extrapolating tail risk.
* **Model Enhancement Pathway**: To capture extreme escalation beyond 7% unemployment, a parametric tail extension (e.g., fitting a generalized additive logit overlay) should be integrated for out-of-sample macroeconomic conditions.

#### **B. Collateral Valuation Lag Dynamics**
The CRE Index edges upward from 2,227.25 to a peak of 2,284.69 in 2027-Q2 before turning downward in late 2027.
* Because the underlying CRE collateral index remains elevated througout 2026-2027, $LGD$ haircuts remain stable.
* Consequently, projected portfolio credit losses across this forecast window are overwhelmingly driven by $PD$ expansion (borrower liquidity distress) rather than $LGD$ expansion (collateral asset value collapse).

#### **C. Capital Planning & Regulatory Takeaways**
1. **Peak Stress Quarters (2026-Q4 / 2027-Q1)**: The portfolio incurs its steepest capital drain during quarters 4 and 5, where quarterly credit losses consume 627.97 bps per quarter.
2. **Stress Capital Buffer (SCB) Implication**: Given the cumulative 9-quarter drawdown of 46.79%, maintaining a post-stress CET1 ratio above the 4.5% regulatory minimum requires a robust baseline pre-stress CET1 buffer. For this uncalibrated synthetic portfolio profile, the baseline capital tier must be calibrated relative to risk-weighted assets (RWAs) to absorb peak drawdown without triggering prompt corrective action.




